In [0]:
# ============================================================
# 02_silver_chicago
# ============================================================
# Source  : bronze.chicago_raw (latest _batch_id)
# Targets : silver.chicago_clean
#           silver.chicago_violations
#           silver.chicago_quarantine

# Write mode: OVERWRITE - source is a full historical extract.
# Every Silver run re-derives from the latest Bronze batch.
# ============================================================


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
if CATALOG in ("spark_catalog"):
    catalogs = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]
    project_cats = [c for c in catalogs
                    if c not in ("spark_catalog",
                                 "system", "samples", "workspace")]
    CATALOG = project_cats[0] if project_cats else CATALOG

BRONZE_TABLE = f"{CATALOG}.bronze.chicago_raw"
SILVER_TABLE = f"{CATALOG}.silver.chicago_clean"
SILVER_VIOLS = f"{CATALOG}.silver.chicago_violations"
QUARANTINE   = f"{CATALOG}.silver.chicago_quarantine"

CHI_ZIP_MIN = 60007
CHI_ZIP_MAX = 60827

print(f"Catalog       : {CATALOG}")
print(f"Bronze source : {BRONZE_TABLE}")
print(f"Silver target : {SILVER_TABLE}")


In [0]:
# Read latest Bronze batch
try:
    spark.table(BRONZE_TABLE).limit(1).collect()
    print(f"Bronze table found: {BRONZE_TABLE}")
except Exception as e:
    raise Exception(f"Bronze table not found: {BRONZE_TABLE}\nError: {e}")

latest_batch = (
    spark.table(BRONZE_TABLE)
    .groupBy("_batch_id")
    .agg(F.max("_extract_ts").alias("ts"))
    .orderBy(F.col("ts").desc())
    .select("_batch_id")
    .first()[0]
)
print(f"Latest batch : {latest_batch}")

df_chi = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("_batch_id") == latest_batch)
    .select(
        "inspection_id", "restaurant_name", "aka_name",
        "license_number", "facility_type", "risk_category",
        "street_address", "city", "state", "zip_code",
        "inspection_date", "inspection_type", "inspection_result",
        "violations_raw", "latitude", "longitude",
        "_batch_id",
        F.col("_extract_ts").alias("_bronze_extract_ts")
    )
    .dropDuplicates(["inspection_id"])
)

print(f"Rows after dedup : {df_chi.count():,}")
print(f"Columns          : {len(df_chi.columns)}")
display(df_chi.limit(3))


In [0]:
# Type casting
df_chi_typed = (
    df_chi
    .withColumn("inspection_date",
        F.to_date(F.col("inspection_date"), "MM/dd/yyyy"))
    .withColumn("latitude",
        F.col("latitude").cast(DoubleType()))
    .withColumn("longitude",
        F.col("longitude").cast(DoubleType()))
    .withColumn("license_number",
        F.col("license_number").cast("long").cast("string"))
)

df_chi_typed = (
    df_chi_typed
    .withColumn("zip_code",
        F.regexp_extract(F.col("zip_code"), r"(\d{5})", 1))
    .withColumn("zip_code",
        F.when(F.col("zip_code") == "", F.lit(None))
         .otherwise(F.col("zip_code").cast("int").cast("string")))
)

print("Type casting complete")
display(df_chi_typed.select(
    "inspection_id", "inspection_date", "zip_code",
    "latitude", "license_number"
).limit(3))


In [0]:
# Field standardisation

# risk_category: 'Risk 1 (High)' → 'High'
df_chi_typed = df_chi_typed.withColumn(
    "risk_category",
    F.when(F.col("risk_category").contains("High"),   F.lit("High"))
     .when(F.col("risk_category").contains("Medium"), F.lit("Medium"))
     .when(F.col("risk_category").contains("Low"),    F.lit("Low"))
     .otherwise(F.lit("Unknown"))
)

# inspection_score derived from inspection_result Assignment mapping: Pass=90, Pass w/ Conditions=80, Fail=70, No Entry=0 allothers=NULL
df_chi_typed = df_chi_typed.withColumn(
    "inspection_score",
    F.when(F.col("inspection_result") == "Pass",               F.lit(90.0))
     .when(F.col("inspection_result") == "Pass w/ Conditions", F.lit(80.0))
     .when(F.col("inspection_result") == "Fail",               F.lit(70.0))
     .when(F.col("inspection_result") == "No Entry",           F.lit(0.0))
     .otherwise(F.lit(None).cast(DoubleType()))
)

# city: real data contains CCHICAGO, CHICAGOChicago, chicago, Chicago Standardise to uppercase 'CHICAGO'
df_chi_typed = (
    df_chi_typed
    .withColumn("city", F.upper(F.trim(F.col("city"))))
    .withColumn("city",
        F.when(F.col("city").isin("CCHICAGO", "CHICAGOCHICAGO"),
               F.lit("CHICAGO"))
         .otherwise(F.col("city")))
)

# facility_type: normalise casing for eg. 'TAVERN' - 'Tavern'
df_chi_typed = df_chi_typed.withColumn(
    "facility_type", F.initcap(F.col("facility_type"))
)

# city_source literal
df_chi_typed = df_chi_typed.withColumn("city_source", F.lit("CHI"))

print("Standardisation complete")
display(df_chi_typed.select(
    "inspection_id", "risk_category", "inspection_score",
    "inspection_result", "city", "facility_type"
).limit(5))


In [0]:
# ── Step 4: DQX Validation ───────────────────────────────────

df_chi_checked = (
    df_chi_typed

    .withColumn("valid_business_name",
        F.col("restaurant_name").isNotNull() &
        (F.col("restaurant_name") != ""))

    .withColumn("valid_inspection_date",
        F.col("inspection_date").isNotNull())

    .withColumn("valid_inspection_type",
        F.col("inspection_type").isNotNull() &
        (F.col("inspection_type") != ""))

    .withColumn("valid_zip_code",
        F.col("zip_code").isNotNull() &
        (F.col("zip_code") != "") &
        (F.col("zip_code") != "null"))

    .withColumn("valid_zip_format",
        F.col("zip_code").rlike(r"^[0-9]+$"))

    .withColumn("valid_zip_range",
        F.when(F.col("zip_code").isNull(), F.lit(False))
         .otherwise(F.col("zip_code").cast("int").between(
             CHI_ZIP_MIN, CHI_ZIP_MAX)))

    .withColumn("valid_results",
        F.col("inspection_result").isNotNull() &
        (F.col("inspection_result") != ""))

    # FIX: was a silent pre-filter — now a proper DQX check
    .withColumn("valid_has_violations",
        F.col("violations_raw").isNotNull() &
        (F.col("violations_raw") != ""))

    # Pass/Pass w/ Conditions must not contain urgent/critical
    .withColumn("valid_no_urgent_pass",
        F.when(
            F.col("violations_raw").isNull() | (F.col("violations_raw") == ""),
            F.lit(True)
        ).otherwise(
            ~(F.col("inspection_result").isin("Pass", "Pass w/ Conditions") &
              F.col("violations_raw").rlike(r"(?i)(urgent|critical)"))
        ))

    .withColumn("_all_checks_pass",
        F.col("valid_business_name")    &
        F.col("valid_inspection_date")  &
        F.col("valid_inspection_type")  &
        F.col("valid_zip_code")         &
        F.col("valid_zip_format")       &
        F.col("valid_zip_range")        &
        F.col("valid_results")          &
        F.col("valid_has_violations")   &
        F.col("valid_no_urgent_pass"))
)

CHECK_COLS = [
    "valid_business_name", "valid_inspection_date",
    "valid_inspection_type", "valid_zip_code", "valid_zip_format",
    "valid_zip_range", "valid_results", "valid_has_violations",
    "valid_no_urgent_pass", "_all_checks_pass"
]

failed_checks_expr = F.concat_ws(",", *[
    F.when(F.col(c) == False, F.lit(c)).otherwise(F.lit(None))
    for c in CHECK_COLS if c != "_all_checks_pass"
])
df_chi_checked = df_chi_checked.withColumn(
    "_dqx_failed_checks", failed_checks_expr
)

df_chi_good = (
    df_chi_checked
    .filter(F.col("_all_checks_pass") == True)
    .drop(*CHECK_COLS, "_dqx_failed_checks")
    .withColumn("_silver_ts",      F.current_timestamp())
    .withColumn("_silver_date",    F.current_date().cast("string"))
    .withColumn("_silver_version", F.lit("v1"))
    .withColumn("_dqx_status",     F.lit("PASS"))
)

df_chi_bad = (
    df_chi_checked
    .filter(
        F.col("_all_checks_pass").isNull() |
        (F.col("_all_checks_pass") == False))
    .withColumn("_silver_ts",      F.current_timestamp())
    .withColumn("_silver_date",    F.current_date().cast("string"))
    .withColumn("_silver_version", F.lit("v1"))
    .withColumn("_dqx_status",     F.lit("FAIL"))
)

original   = df_chi_typed.count()
good_count = df_chi_good.count()
bad_count  = df_chi_bad.count()
lost       = original - (good_count + bad_count)

print(f"Original rows : {original:,}")
print(f"GOOD rows     : {good_count:,}")
print(f"BAD rows      : {bad_count:,}")
print(f"Rows lost     : {lost}  <- must be 0")
assert lost == 0, f"Lost {lost} rows — logic error in DQX split"

print("\nFailed check breakdown (quarantine):")
display(
    df_chi_bad
    .groupBy("_dqx_failed_checks")
    .count()
    .orderBy(F.desc("count"))
    .limit(15)
)


In [0]:
# Violation parsing

df_chi_viols = (
    df_chi_good
    .filter(F.col("violations_raw").isNotNull())
    .withColumn("viol_entry",
        F.explode(F.split(F.col("violations_raw"), r"\|")))
    .withColumn("viol_entry", F.trim(F.col("viol_entry")))
    .filter(F.col("viol_entry") != "")

    .withColumn("violation_code",
        F.regexp_extract(F.col("viol_entry"), r"^(\d+)\.", 1))
    .withColumn("violation_description",
        F.trim(F.regexp_extract(F.col("viol_entry"),
            r"^\d+\.\s*(.+?)(?:\s*-\s*Comments:.*)?$", 1)))
    .withColumn("inspector_comment",
        F.regexp_extract(F.col("viol_entry"),
            r"-\s*Comments:\s*(.+)", 1))

    .filter(F.col("violation_code") != "")

    # One row per inspection per unique violation code
    .dropDuplicates(["inspection_id", "violation_code"])

    .select(
        "inspection_id", "restaurant_name",
        "inspection_date", "city_source",
        "violation_code", "violation_description",
        "inspector_comment", "_batch_id",
        F.lit(None).cast(DoubleType()).alias("violation_points"),
        F.lit(None).cast("string").alias("violation_detail"),
        F.lit(None).cast(IntegerType()).alias("violation_slot")
    )
)

# Count distinct violations per inspection
viol_counts = (
    df_chi_viols
    .groupBy("inspection_id")
    .agg(F.countDistinct("violation_code").alias("violation_count"))
)

df_chi_good = (
    df_chi_good.alias("g")
    .join(viol_counts.alias("v"), on="inspection_id", how="left")
    .withColumn("violation_count",
        F.coalesce(F.col("v.violation_count"), F.lit(0)).cast(IntegerType()))
    .select(
        *[F.col(f"g.{c}") for c in df_chi_good.columns],
        F.col("violation_count")
    )
)

print(f"Clean inspections (with violation count) : {df_chi_good.count():,}")
print(f"Violation rows parsed                    : {df_chi_viols.count():,}")
display(df_chi_viols.select(
    "inspection_id", "violation_code",
    "violation_description", "inspector_comment"
).limit(5))


In [0]:
# Step 7: Verification
display(spark.sql(f"""
    SELECT
        city_source,
        COUNT(*)                               AS total_inspections,
        ROUND(AVG(inspection_score), 2)        AS avg_score,
        ROUND(AVG(violation_count), 2)         AS avg_violations,
        SUM(CASE WHEN inspection_result = 'Pass'               THEN 1 ELSE 0 END) AS pass_count,
        SUM(CASE WHEN inspection_result = 'Fail'               THEN 1 ELSE 0 END) AS fail_count,
        SUM(CASE WHEN inspection_result = 'Pass w/ Conditions' THEN 1 ELSE 0 END) AS pass_cond,
        SUM(CASE WHEN risk_category = 'High'    THEN 1 ELSE 0 END) AS high_risk,
        SUM(CASE WHEN risk_category = 'Medium'  THEN 1 ELSE 0 END) AS medium_risk,
        SUM(CASE WHEN risk_category = 'Low'     THEN 1 ELSE 0 END) AS low_risk,
        SUM(CASE WHEN risk_category = 'Unknown' THEN 1 ELSE 0 END) AS unknown_risk
    FROM {SILVER_TABLE}
    GROUP BY city_source
"""))

display(spark.sql(f"""
    SELECT
        SUM(CASE WHEN valid_business_name   = false THEN 1 ELSE 0 END) AS failed_name,
        SUM(CASE WHEN valid_inspection_date = false THEN 1 ELSE 0 END) AS failed_date,
        SUM(CASE WHEN valid_inspection_type = false THEN 1 ELSE 0 END) AS failed_type,
        SUM(CASE WHEN valid_zip_code        = false THEN 1 ELSE 0 END) AS failed_zip_null,
        SUM(CASE WHEN valid_zip_format      = false THEN 1 ELSE 0 END) AS failed_zip_fmt,
        SUM(CASE WHEN valid_zip_range       = false THEN 1 ELSE 0 END) AS failed_zip_range,
        SUM(CASE WHEN valid_results         = false THEN 1 ELSE 0 END) AS failed_results,
        SUM(CASE WHEN valid_no_urgent_pass  = false THEN 1 ELSE 0 END) AS failed_urgent,
        COUNT(*)                                                         AS total_quarantined
    FROM {QUARANTINE}
"""))
display(spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}"))
